# EfficientNet-B0 + MIL com Optuna — 49 patches (BCEFocalOrdinalLoss)

Réplica de `b0-mil-focal.ipynb` com bags de **49 patches** (vs 36 do original).

- **MIL (Gated Attention pooling)** sobre bags de 49 patches histopatológicos
- **BCEFocalOrdinalLoss** combinando focal loss + penalidade ordinal
- **Optuna** para busca dos pesos ótimos da perda (`gamma`, `w_focal`, `w_ord`, `lr`)
- Filtragem por entropia (remove 20% mais difíceis)
- Treino completo: 50 épocas, patience 10, avaliação no teste

**Hipótese**: bags maiores (49 patches, cobertura ~7×7) capturam mais contexto espacial
e podem superar o QWK=0.86 do b0-mil-focal com 36 patches.

In [ ]:
import sys
sys.path.append('../../../')

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler, SequentialSampler
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import albumentations as Albu
from warmup_scheduler import GradualWarmupScheduler
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, recall_score, precision_score
from tqdm import tqdm
import optuna
from optuna.pruners import MedianPruner
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

from utils.mil import PandasWithMilDataset, EfficientNetMIL

## Configuração

In [ ]:
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Hiperparâmetros fixos
# BATCH_SIZE reduzido de 8 → 6: bags de 49 patches ocupam ~36% mais VRAM que 36 patches
BATCH_SIZE     = 6
NUM_WORKERS    = 4
OUTPUT_CLASSES = 5
WEIGHT_DECAY   = 1e-4
WARMUP_FACTOR  = 2
WARMUP_EPOCHS  = 1
N_EPOCHS       = 50
DROPOUT_RATE   = 0.4
PATIENCE       = 10
MAX_PATCHES    = 49    # ← diferença principal em relação ao b0-mil-focal (36)
GRAD_CLIP      = 1.0
AMP_DTYPE      = torch.float16

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Optuna — subset + menos patches por trial para reduzir custo
OPTUNA_BATCH_SIZE  = 6
OPTUNA_MAX_PATCHES = 16    # reduz ~3× os forwards por bag durante busca
OPTUNA_SUBSET_FRAC = 0.25
N_OPTUNA_TRIALS    = 30
N_OPTUNA_EPOCHS    = 5

ROOT_DIR   = '../../..'
IMAGES_DIR = '/home/woshington/Projects/Doutorado/bag_of_patches'

os.makedirs('logs',   exist_ok=True)
os.makedirs('models', exist_ok=True)

MODEL_PATH = 'models/b0-mil-focal-49-optuna.pth'
LOG_PATH   = 'logs/b0-mil-focal-49-optuna.txt'
STUDY_DB   = 'sqlite:///logs/b0-mil-focal-49-optuna.db'
STUDY_NAME = 'b0-mil-focal-49-optuna'

print(f'BATCH_SIZE={BATCH_SIZE} | MAX_PATCHES={MAX_PATCHES} | AMP={AMP_DTYPE}')
print(f'Optuna: SUBSET={OPTUNA_SUBSET_FRAC:.0%} | MAX_PATCHES={OPTUNA_MAX_PATCHES} | EPOCHS={N_OPTUNA_EPOCHS}')

## Função de Perda

In [ ]:
class BCEFocalOrdinalLoss(nn.Module):
    """
    Perda combinada: Focal Loss + Penalidade Ordinal.

    loss = w_focal * focal + w_ord * ordinal_mse
    """

    def __init__(self, gamma: float = 2.0, w_focal: float = 1.0, w_ord: float = 1.0):
        super().__init__()
        self.gamma   = gamma
        self.w_focal = w_focal
        self.w_ord   = w_ord

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        logits  = logits.float()
        targets = targets.to(logits.device).float()
        probs   = torch.sigmoid(logits)

        bce        = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t        = probs * targets + (1 - probs) * (1 - targets)
        focal_loss = ((1 - p_t) ** self.gamma * bce).mean()

        expected_cls = probs.sum(dim=1)
        target_cls   = targets.sum(dim=1)
        max_cls      = logits.shape[1]
        ord_loss     = ((expected_cls - target_cls) ** 2).mean() / (max_cls ** 2)

        return self.w_focal * focal_loss + self.w_ord * ord_loss


print('BCEFocalOrdinalLoss definida.')

## Carregamento de Dados

In [ ]:
def remove_nonexistent(df, images_dir):
    mask = df['image_id'].apply(lambda x: os.path.isdir(os.path.join(images_dir, x)))
    return df[mask].reset_index(drop=True)


df_all = pd.read_csv(f'{ROOT_DIR}/data/train_5fold.csv')
df_all.columns = df_all.columns.str.strip()

# Filtragem por entropia (remove 20% mais difíceis)
df_entropy = pd.read_csv(f'{ROOT_DIR}/data/entropy.csv')
df_entropy = df_entropy.sort_values('difficulty_score', ascending=False)
n_remove   = int(len(df_entropy) * 0.2)
ids_remove = set(df_entropy.head(n_remove)['image_id'])
df_all     = df_all[~df_all['image_id'].isin(ids_remove)].reset_index(drop=True)

train_idx = np.where(df_all['fold'] != 3)[0]
valid_idx = np.where(df_all['fold'] == 3)[0]

df_train = df_all.loc[train_idx].reset_index(drop=True)
df_val   = df_all.loc[valid_idx].reset_index(drop=True)
df_test  = pd.read_csv(f'{ROOT_DIR}/data/test.csv')
df_test.columns = df_test.columns.str.strip()

df_train = remove_nonexistent(df_train, IMAGES_DIR)
df_val   = remove_nonexistent(df_val,   IMAGES_DIR)
df_test  = remove_nonexistent(df_test,  IMAGES_DIR)

print(f'Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')
print('Distribuição de classes (train):')
print(df_train['isup_grade'].value_counts().sort_index())

## Augmentação

In [ ]:
train_transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
    Albu.RandomRotate90(p=0.5),
    Albu.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = Albu.Compose([
    Albu.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

## Modelo

In [ ]:
def build_model():
    backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    model = EfficientNetMIL(
        model=backbone,
        output_classes=OUTPUT_CLASSES,
        fine_tune=150,
        dropout_rate=DROPOUT_RATE,
        hidden_dim=512,
        gated=True,
        pool='att',
    )
    return model.to(device)


m = build_model()
trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
total     = sum(p.numel() for p in m.parameters())
print(f'EfficientNet-B0 MIL — params treináveis: {trainable:,} / {total:,}')
del m
torch.cuda.empty_cache()

## Funções de Treino e Validação

In [ ]:
def training_step(model, dataloader, optimizer, device, loss_fn, scaler, grad_clip=1.0):
    model.train()
    losses = []
    bar = tqdm(dataloader, desc='Training', leave=False)
    for bag, mask, targets, _ in bar:
        bag     = bag.to(device, non_blocking=True)
        mask    = mask.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
            out  = model(bag, mask)
            loss = loss_fn(out['logits'], targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.detach().cpu().item())
        bar.set_postfix(loss=f'{losses[-1]:.5f}')
    return losses


def validation_step(model, dataloader, device, loss_fn):
    model.eval()
    val_loss, all_preds, all_targets = [], [], []
    with torch.no_grad():
        for bag, mask, targets, _ in dataloader:
            bag     = bag.to(device, non_blocking=True)
            mask    = mask.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                out  = model(bag, mask)
                loss = loss_fn(out['logits'], targets)
            probs       = torch.sigmoid(out['logits'])
            preds       = (probs > 0.5).sum(dim=1)
            targets_cls = targets.sum(dim=1).long()
            all_preds.append(preds.cpu())
            all_targets.append(targets_cls.cpu())
            val_loss.append(loss.cpu().item())
    all_preds   = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    return {
        'val_loss':      np.mean(val_loss),
        'val_acc':       accuracy_score(all_targets, all_preds),
        'val_kappa':     cohen_kappa_score(all_targets, all_preds, weights='quadratic'),
        'val_f1':        f1_score(all_targets, all_preds, average='macro', zero_division=0),
        'val_recall':    recall_score(all_targets, all_preds, average='macro', zero_division=0),
        'val_precision': precision_score(all_targets, all_preds, average='macro', zero_division=0),
    }

## Busca de Hiperparâmetros com Optuna

O Optuna busca `gamma`, `w_focal`, `w_ord` e `lr` da `BCEFocalOrdinalLoss`.
Para reduzir o custo por trial:
- **25% do train** (`OPTUNA_SUBSET_FRAC`) — 4× menos batches/época
- **16 patches/bag** (`OPTUNA_MAX_PATCHES`) — ~3× menos forwards/batch vs 49
- **5 épocas** por trial (`N_OPTUNA_EPOCHS`)

In [ ]:
rng_optuna = np.random.default_rng(SEED)
optuna_idx = rng_optuna.choice(len(df_train), size=int(len(df_train) * OPTUNA_SUBSET_FRAC), replace=False)
df_optuna  = df_train.iloc[optuna_idx].reset_index(drop=True)

optuna_train_ds = PandasWithMilDataset(
    IMAGES_DIR, df_optuna, transforms=train_transforms,
    normalize=False, max_patches=OPTUNA_MAX_PATCHES
)
optuna_val_ds = PandasWithMilDataset(
    IMAGES_DIR, df_val, transforms=val_transforms,
    normalize=False, max_patches=OPTUNA_MAX_PATCHES
)

optuna_train_loader = DataLoader(
    optuna_train_ds, batch_size=OPTUNA_BATCH_SIZE, num_workers=NUM_WORKERS,
    sampler=RandomSampler(optuna_train_ds),
    pin_memory=True, drop_last=True,
)
optuna_val_loader = DataLoader(
    optuna_val_ds, batch_size=OPTUNA_BATCH_SIZE * 2, num_workers=NUM_WORKERS,
    sampler=SequentialSampler(optuna_val_ds),
    pin_memory=True,
)

print(f'Optuna subset: {len(df_optuna)} imgs ({OPTUNA_SUBSET_FRAC:.0%}) | {OPTUNA_MAX_PATCHES} patches/bag')
print(f'Train batches: {len(optuna_train_loader)} | Val batches: {len(optuna_val_loader)}')

In [ ]:
def objective(trial: optuna.Trial) -> float:
    gamma   = trial.suggest_float('gamma',   0.5, 3.0)
    w_focal = trial.suggest_float('w_focal', 0.1, 2.0)
    w_ord   = trial.suggest_float('w_ord',   0.1, 2.0)
    lr      = trial.suggest_float('lr',      1e-5, 1e-3, log=True)

    model   = build_model()
    loss_fn = BCEFocalOrdinalLoss(gamma=gamma, w_focal=w_focal, w_ord=w_ord)
    opt     = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scaler  = torch.amp.GradScaler()

    best_kappa = 0.0
    for epoch in range(N_OPTUNA_EPOCHS):
        training_step(model, optuna_train_loader, opt, device, loss_fn, scaler, grad_clip=GRAD_CLIP)
        metrics = validation_step(model, optuna_val_loader, device, loss_fn)
        kappa   = metrics['val_kappa']

        trial.report(kappa, epoch)
        if trial.should_prune():
            del model
            torch.cuda.empty_cache()
            raise optuna.exceptions.TrialPruned()

        best_kappa = max(best_kappa, kappa)

    del model
    torch.cuda.empty_cache()
    return best_kappa


print('objective definido.')

## Busca Optuna

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=2),
    storage=STUDY_DB,
    load_if_exists=True,
)

study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

best = study.best_trial
print(f'\nMelhor trial: #{best.number}')
print(f'  Kappa:   {best.value:.4f}')
for k, v in best.params.items():
    print(f'  {k}: {v}')

In [ ]:
importances = optuna.importance.get_param_importances(study)
print('Importância dos hiperparâmetros:')
for param, imp in importances.items():
    print(f'  {param}: {imp:.4f}')

best_trial   = study.best_trial
BEST_GAMMA   = best_trial.params['gamma']
BEST_W_FOCAL = best_trial.params['w_focal']
BEST_W_ORD   = best_trial.params['w_ord']
BEST_LR      = best_trial.params['lr']

print(f'\nMelhores hiperparâmetros:')
print(f'  gamma={BEST_GAMMA:.4f}, w_focal={BEST_W_FOCAL:.4f}, w_ord={BEST_W_ORD:.4f}, lr={BEST_LR:.2e}')

## Visualizações Optuna

In [ ]:
from optuna import visualization as optvis

fig = optvis.plot_optimization_history(study)
fig.update_layout(title='Histórico de Otimização — Kappa por Trial', height=450)
fig.show()

In [ ]:
fig = optvis.plot_intermediate_values(study)
fig.update_layout(title='Valores Intermediários por Época (trials com pruning)', height=450)
fig.show()

In [ ]:
fig = optvis.plot_parallel_coordinate(study, params=['gamma', 'w_focal', 'w_ord', 'lr'])
fig.update_layout(title='Coordenadas Paralelas — Hiperparâmetros vs Kappa', height=500)
fig.show()

In [ ]:
fig = optvis.plot_param_importances(study)
fig.update_layout(title='Importância dos Hiperparâmetros (fANOVA)', height=400)
fig.show()

In [ ]:
print(f'Storage: {STUDY_DB}')
print(f'Study  : {STUDY_NAME}')
print('\nPara abrir o dashboard, execute no terminal:')
print(f'  optuna-dashboard {STUDY_DB}')

## Treino Completo com Melhores Hiperparâmetros

Usa `MAX_PATCHES=49` (cobertura 7×7), dataset completo, 50 épocas e patience 10.

In [ ]:
train_dataset = PandasWithMilDataset(
    IMAGES_DIR, df_train, transforms=train_transforms,
    normalize=False, max_patches=MAX_PATCHES
)
valid_dataset = PandasWithMilDataset(
    IMAGES_DIR, df_val, transforms=val_transforms,
    normalize=False, max_patches=MAX_PATCHES
)
test_dataset = PandasWithMilDataset(
    IMAGES_DIR, df_test, transforms=val_transforms,
    normalize=False, max_patches=MAX_PATCHES
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    sampler=RandomSampler(train_dataset),
    pin_memory=True, prefetch_factor=2, persistent_workers=True, drop_last=True,
)
valid_loader = DataLoader(
    valid_dataset, batch_size=BATCH_SIZE * 2, num_workers=NUM_WORKERS,
    sampler=SequentialSampler(valid_dataset),
    pin_memory=True, prefetch_factor=2, persistent_workers=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE * 2, num_workers=NUM_WORKERS,
    shuffle=False, pin_memory=True, prefetch_factor=2, persistent_workers=True,
)

print(f'Train: {len(train_loader)} batches | Val: {len(valid_loader)} batches | Test: {len(test_loader)} batches')

In [ ]:
model = build_model()

loss_function = BCEFocalOrdinalLoss(
    gamma=BEST_GAMMA,
    w_focal=BEST_W_FOCAL,
    w_ord=BEST_W_ORD,
)

optimizer = optim.AdamW(
    model.parameters(),
    lr=BEST_LR / WARMUP_FACTOR,
    weight_decay=WEIGHT_DECAY,
)

scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, N_EPOCHS - WARMUP_EPOCHS
)
scheduler = GradualWarmupScheduler(
    optimizer,
    multiplier=WARMUP_FACTOR,
    total_epoch=WARMUP_EPOCHS,
    after_scheduler=scheduler_cosine,
)

scaler = torch.amp.GradScaler()

print(f'Modelo: EfficientNet-B0 + MIL (GatedAttention) | max_patches={MAX_PATCHES}')
print(f'Perda : gamma={BEST_GAMMA:.4f}, w_focal={BEST_W_FOCAL:.4f}, w_ord={BEST_W_ORD:.4f}')
print(f'Opt   : AdamW | lr={BEST_LR:.2e} | wd={WEIGHT_DECAY:.1e}')
print(f'Treino: {N_EPOCHS} épocas | patience={PATIENCE} | batch={BATCH_SIZE}')

In [ ]:
best_kappa = 0.0
best_epoch = 0
no_improve = 0

history = {
    'train_loss':    [],
    'val_loss':      [],
    'val_acc':       [],
    'val_kappa':     [],
    'val_f1':        [],
    'val_recall':    [],
    'val_precision': [],
}

with open(LOG_PATH, 'a') as f:
    f.write(f'max_patches={MAX_PATCHES} | batch={BATCH_SIZE}\n')
    f.write(f'Optuna best: gamma={BEST_GAMMA:.4f}, w_focal={BEST_W_FOCAL:.4f}, '
            f'w_ord={BEST_W_ORD:.4f}, lr={BEST_LR:.2e}\n')

print('Iniciando treino completo...')
print('=' * 80)

for epoch in range(1, N_EPOCHS + 1):
    print(f'\nÉpoca {epoch}/{N_EPOCHS}')

    train_losses = training_step(model, train_loader, optimizer, device, loss_function, scaler, grad_clip=GRAD_CLIP)
    metrics      = validation_step(model, valid_loader, device, loss_function)

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    history['train_loss'].append(np.mean(train_losses))
    for k in ['val_loss', 'val_acc', 'val_kappa', 'val_f1', 'val_recall', 'val_precision']:
        history[k].append(metrics[k])

    print(f'  Train Loss: {history["train_loss"][-1]:.5f}')
    print(f'  Val   Loss: {metrics["val_loss"]:.5f} | Acc: {metrics["val_acc"]*100:.2f}% | Kappa: {metrics["val_kappa"]:.4f} | F1: {metrics["val_f1"]:.4f}')
    print(f'  LR: {current_lr:.2e}')

    log_line = (f'epoch: {epoch} | lr: {current_lr:.2e} | '
                f'train_loss: {history["train_loss"][-1]:.5f} | '
                f'val_loss: {metrics["val_loss"]:.5f} | '
                f'val_acc: {metrics["val_acc"]:.4f} | '
                f'val_kappa: {metrics["val_kappa"]:.4f}\n')
    with open(LOG_PATH, 'a') as f:
        f.write(log_line)

    if metrics['val_kappa'] > best_kappa:
        best_kappa = metrics['val_kappa']
        best_epoch = epoch
        no_improve = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  Melhor modelo salvo! Kappa: {best_kappa:.4f}')
    else:
        no_improve += 1
        print(f'  Sem melhora por {no_improve} época(s)')

    if no_improve >= PATIENCE:
        print(f'\nEarly stopping na época {epoch}. Melhor: época {best_epoch} (Kappa={best_kappa:.4f})')
        break

print('\nTreino concluído!')
print(f'Melhor Kappa de validação: {best_kappa:.4f} na época {best_epoch}')

## Curvas de Aprendizado

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(history['train_loss'], label='Train Loss')
axes[0, 0].plot(history['val_loss'],   label='Val Loss')
axes[0, 0].set_title(f'Loss — B0 MIL Focal ({MAX_PATCHES} patches)')
axes[0, 0].set_xlabel('Época')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(history['val_acc'], color='green', label='Val Accuracy')
axes[0, 1].set_title('Acurácia de Validação')
axes[0, 1].set_xlabel('Época')
axes[0, 1].legend()
axes[0, 1].grid(True)

axes[1, 0].plot(history['val_kappa'], color='orange', label='Val Kappa')
axes[1, 0].axhline(y=0.86, color='red', linestyle='--', label='B0-MIL-36: 0.86')
axes[1, 0].set_title('Kappa Quadrático de Validação')
axes[1, 0].set_xlabel('Época')
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].plot(history['val_f1'], color='red', label='Val F1')
axes[1, 1].set_title('F1 Macro de Validação')
axes[1, 1].set_xlabel('Época')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('logs/b0-mil-focal-49-optuna-training.png', dpi=300, bbox_inches='tight')
plt.show()

## Avaliação no Conjunto de Teste

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
model.eval()

all_preds, all_targets = [], []

with torch.no_grad():
    for bag, mask, targets, _ in tqdm(test_loader, desc='Testing'):
        bag  = bag.to(device,  non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
            out = model(bag, mask)
        probs       = torch.sigmoid(out['logits'])
        preds       = (probs > 0.5).sum(dim=1)
        targets_cls = targets.sum(dim=1).long()
        all_preds.append(preds.cpu())
        all_targets.append(targets_cls.cpu())

all_preds   = torch.cat(all_preds).numpy()
all_targets = torch.cat(all_targets).numpy()

# ── Bootstrap CI (1 000 resamples) ───────────────────────────────────────────
N_BOOTSTRAP = 1000
rng = np.random.default_rng(SEED)
n   = len(all_targets)
boot_acc   = np.empty(N_BOOTSTRAP)
boot_kappa = np.empty(N_BOOTSTRAP)
boot_f1    = np.empty(N_BOOTSTRAP)

for i in tqdm(range(N_BOOTSTRAP), desc='Bootstrap'):
    idx           = rng.integers(0, n, size=n)
    boot_acc[i]   = accuracy_score(all_targets[idx], all_preds[idx])
    boot_kappa[i] = cohen_kappa_score(all_targets[idx], all_preds[idx], weights='quadratic')
    boot_f1[i]    = f1_score(all_targets[idx], all_preds[idx], average='macro', zero_division=0)

def boot_stats(arr):
    return arr.std(ddof=1), np.percentile(arr, 2.5), np.percentile(arr, 97.5)

test_acc   = accuracy_score(all_targets, all_preds)
test_kappa = cohen_kappa_score(all_targets, all_preds, weights='quadratic')
test_f1    = f1_score(all_targets, all_preds, average='macro', zero_division=0)

acc_std,   acc_lo,   acc_hi   = boot_stats(boot_acc)
kappa_std, kappa_lo, kappa_hi = boot_stats(boot_kappa)
f1_std,    f1_lo,    f1_hi    = boot_stats(boot_f1)

print('=' * 70)
print(f'RESULTADOS NO TESTE — EfficientNet-B0 MIL ({MAX_PATCHES} patches)')
print('=' * 70)
print(f'Accuracy : {test_acc*100:.2f}% ± {acc_std*100:.2f}%  [{acc_lo*100:.2f}% – {acc_hi*100:.2f}%]')
print(f'QW Kappa : {test_kappa:.4f} ± {kappa_std:.4f}  [{kappa_lo:.4f} – {kappa_hi:.4f}]')
print(f'Macro F1 : {test_f1:.4f} ± {f1_std:.4f}  [{f1_lo:.4f} – {f1_hi:.4f}]')
print('=' * 70)
print(f'Melhores params: gamma={BEST_GAMMA:.4f}, w_focal={BEST_W_FOCAL:.4f}, w_ord={BEST_W_ORD:.4f}, lr={BEST_LR:.2e}')

from sklearn.metrics import classification_report
print('\nClassification Report:')
print(classification_report(all_targets, all_preds,
      target_names=[f'ISUP {i}' for i in range(6)], digits=4, zero_division=0))

with open(LOG_PATH, 'a') as f:
    f.write('\n=== TEST RESULTS ===\n')
    f.write(f'Bootstrap resamples: {N_BOOTSTRAP}\n')
    f.write(f'acc={test_acc:.4f} ± {acc_std:.4f}  [{acc_lo:.4f} – {acc_hi:.4f}]\n')
    f.write(f'kappa={test_kappa:.4f} ± {kappa_std:.4f}  [{kappa_lo:.4f} – {kappa_hi:.4f}]\n')
    f.write(f'f1={test_f1:.4f} ± {f1_std:.4f}  [{f1_lo:.4f} – {f1_hi:.4f}]\n')
    f.write(f'gamma={BEST_GAMMA:.4f} | w_focal={BEST_W_FOCAL:.4f} | w_ord={BEST_W_ORD:.4f} | lr={BEST_LR:.2e}\n')

In [ ]:
cm      = confusion_matrix(all_targets, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
classes = ['ISUP 0', 'ISUP 1', 'ISUP 2', 'ISUP 3', 'ISUP 4', 'ISUP 5']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=classes, yticklabels=classes, ax=axes[0],
)
axes[0].set_title(f'Matriz de Confusão — B0 MIL ({MAX_PATCHES} patches)')
axes[0].set_ylabel('Classe Verdadeira')
axes[0].set_xlabel('Classe Prevista')

sns.heatmap(
    cm_norm, annot=True, fmt='.3f', cmap='Blues',
    xticklabels=classes, yticklabels=classes, ax=axes[1],
)
axes[1].set_title('Matriz de Confusão Normalizada')
axes[1].set_ylabel('Classe Verdadeira')
axes[1].set_xlabel('Classe Prevista')

plt.tight_layout()
plt.savefig('logs/b0-mil-focal-49-optuna-confusion-matrix.png', dpi=300, bbox_inches='tight')
plt.show()